# Student Workbook: Visual Linear Regression

This is your copy of the class. Same data, same charts, same order — but the core calculations are left for you to fill in. Nothing here is graded. The point is simply to rebuild, in your own hands, what we did together in class.

Each exercise looks like this:

> ### ✏️ Try it yourself
> A short instruction.

followed by a blank code cell for you to fill in, and a collapsed **Show solution** block you can open if you get stuck or want to check your work.

### The opening question

> We have information about how many hours students studied and their exam scores. If another student studies **6.5 hours**, how could we estimate their score?

Keep this in mind — we'll answer it properly, and *see* it on a chart, by the end.

## Section 1 — Start With Data

In [1]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go

pd.set_option("display.precision", 2)

In [2]:
scores_1d = pd.read_csv("../data/study_scores_1d.csv")
scores_1d

,hours_studied,exam_score
0,1.8,47.1
1,2.0,40.2
2,2.8,55.7
3,4.0,56.7
4,4.5,58.6
5,4.5,55.7
6,4.6,67.6
7,6.2,68.2
8,6.6,68.7
9,7.1,71.9


In [3]:
figure = go.Figure()
figure.add_trace(
    go.Scatter(
        x=scores_1d["hours_studied"], y=scores_1d["exam_score"], mode="markers",
        marker=dict(size=11, color="#2563eb"), name="students",
        hovertemplate="Hours studied: %{x}<br>Exam score: %{y}<extra></extra>",
    )
)
figure.update_layout(
    title="Hours Studied vs Exam Score", xaxis_title="Hours studied", yaxis_title="Exam score",
    template="plotly_white", width=750, height=500,
)
figure.show()

> ### 💭 Think about it
>
> - Do you see a relationship?
> - If someone studies more, what generally seems to happen?
> - Could one line summarize this relationship?

Before fitting anything, let's also look at `hours_studied` on its own — how spread out is it?

In [4]:
histogram_figure = go.Figure()
histogram_figure.add_trace(
    go.Histogram(x=scores_1d["hours_studied"], marker=dict(color="#2563eb"))
)
histogram_figure.update_layout(
    title="Distribution of Hours Studied",
    xaxis_title="Hours studied", yaxis_title="Number of students",
    template="plotly_white", width=750, height=400,
)
histogram_figure.show()

> ### 💭 Think about it
>
> - Are study hours spread evenly, or clustered somewhere?

## Section 2 — Human Regression

Our line has the form:

$$\hat y = mx + b$$

```text
m = how steep the line is   (the slope)
b = where the line starts   (the intercept, the value of y when x = 0)
```

Here's a helper function so you can try lines without redrawing the chart from scratch every time.

In [5]:
def plot_line_over_data(dataframe, slope, intercept, title="Try your own line"):
    x = dataframe["hours_studied"]
    y = dataframe["exam_score"]

    line_x = np.linspace(x.min() - 0.5, x.max() + 0.5, 50)
    line_y = slope * line_x + intercept

    figure = go.Figure()
    figure.add_trace(
        go.Scatter(
            x=x, y=y, mode="markers", marker=dict(size=11, color="#2563eb"),
            name="actual students", hovertemplate="Hours: %{x}<br>Score: %{y}<extra></extra>",
        )
    )
    figure.add_trace(
        go.Scatter(
            x=line_x, y=line_y, mode="lines", line=dict(color="#dc2626", width=3),
            name=f"y = {slope:.1f}x + {intercept:.1f}",
        )
    )
    figure.update_layout(
        title=title, xaxis_title="Hours studied", yaxis_title="Exam score",
        template="plotly_white", width=750, height=500,
    )
    return figure

In [6]:
# Change these two numbers and rerun this cell as many times as you like.
slope = 5.0
intercept = 35.0

plot_line_over_data(scores_1d, slope, intercept).show()

> ### 📝 Note
>
> Suggested ranges for this dataset: slope m from -2 to 12, intercept b from 0 to 70.

## Section 3 — Manual Line-Fitting Challenge

> **Your challenge:** find a line that visually represents the points as well as possible. Use only your eyes for now.

In [7]:
plot_line_over_data(scores_1d, slope=0, intercept=60, title="m = 0, b = 60").show()

> ### 💭 Think about it
>
> - What's wrong with this line?

In [8]:
plot_line_over_data(scores_1d, slope=10, intercept=10, title="m = 10, b = 10").show()

> ### 💭 Think about it
>
> - What's wrong with this one? Is it wrong in the same way as before?

In [9]:
# Your turn: pick numbers you think look good.
my_slope = 6.0
my_intercept = 30.0

plot_line_over_data(scores_1d, my_slope, my_intercept, title="My attempt").show()

> ### 📝 Note
>
> Different people will land on different (m, b) pairs. That's expected — and it's exactly why we need
> an objective way to measure "better," which is what the next few sections build.

## Section 4 — Predictions

Let's pick one concrete candidate line:

$$\hat y = 5x + 35$$

$\hat y$ ("y hat") means **the model's prediction** — as opposed to $y$, the actual observed score.

### ✏️ Try it yourself

Using `candidate_slope = 5.0` and `candidate_intercept = 35.0`, compute the predicted score for every row in `scores_1d` and store it in a new column called `predicted_score`.

In [10]:
candidate_slope = 5.0
candidate_intercept = 35.0

# TODO: compute predicted_score for every row and add it as a new column

<details>
<summary>Show solution</summary>

```python
candidate_slope = 5.0
candidate_intercept = 35.0

scores_1d["predicted_score"] = candidate_slope * scores_1d["hours_studied"] + candidate_intercept
scores_1d
```

</details>

Continuing with the worked values so the rest of the notebook has something to plot:

In [11]:
candidate_slope = 5.0
candidate_intercept = 35.0
scores_1d["predicted_score"] = candidate_slope * scores_1d["hours_studied"] + candidate_intercept

example_row = scores_1d.iloc[5]

figure = plot_line_over_data(scores_1d, candidate_slope, candidate_intercept, title="One prediction, highlighted")
figure.add_trace(
    go.Scatter(
        x=[example_row["hours_studied"]], y=[example_row["exam_score"]],
        mode="markers", marker=dict(size=16, color="#16a34a", symbol="circle"), name="Actual score",
    )
)
figure.add_trace(
    go.Scatter(
        x=[example_row["hours_studied"]], y=[example_row["predicted_score"]],
        mode="markers", marker=dict(size=16, color="#dc2626", symbol="x"), name="Predicted score",
    )
)
figure.show()

## Section 5 — The Error

$$e_i = y_i - \hat y_i$$

### ✏️ Try it yourself

Add an `error` column to `scores_1d` equal to `exam_score - predicted_score`.

In [12]:
# TODO: compute the error column

<details>
<summary>Show solution</summary>

```python
scores_1d["error"] = scores_1d["exam_score"] - scores_1d["predicted_score"]
scores_1d[["hours_studied", "exam_score", "predicted_score", "error"]].round(2)
```

</details>

In [13]:
scores_1d["error"] = scores_1d["exam_score"] - scores_1d["predicted_score"]

figure = plot_line_over_data(scores_1d, candidate_slope, candidate_intercept, title="Errors as vertical segments")
for _, row in scores_1d.iterrows():
    figure.add_trace(
        go.Scatter(
            x=[row["hours_studied"], row["hours_studied"]], y=[row["predicted_score"], row["exam_score"]],
            mode="lines", line=dict(color="#f59e0b", width=2, dash="dot"), showlegend=False,
        )
    )
figure.show()

scores_1d[["hours_studied", "exam_score", "predicted_score", "error"]].round(2)

,hours_studied,exam_score,predicted_score,error
0,1.8,47.1,44.0,3.1
1,2.0,40.2,45.0,-4.8
2,2.8,55.7,49.0,6.7
3,4.0,56.7,55.0,1.7
4,4.5,58.6,57.5,1.1
5,4.5,55.7,57.5,-1.8
6,4.6,67.6,58.0,9.6
7,6.2,68.2,66.0,2.2
8,6.6,68.7,68.0,0.7
9,7.1,71.9,70.5,1.4


> ### 💭 Think about it
>
> - If we want one number describing how bad the entire line is, why can't we just add all these errors?

In [14]:
error_a = 4
error_b = -4
print("error_a + error_b =", error_a + error_b, "-- but both predictions were wrong!")

error_a + error_b = 0 -- but both predictions were wrong!


## Section 6 — Squaring the Error

$$e_i^2 = (y_i - \hat y_i)^2$$

Squaring fixes cancellation: negatives become positive, errors can't cancel, and big mistakes are penalized much more than small ones.

### ✏️ Try it yourself

Add a `squared_error` column: the `error` column, squared.

In [15]:
# TODO: compute the squared_error column

<details>
<summary>Show solution</summary>

```python
scores_1d["squared_error"] = scores_1d["error"] ** 2
scores_1d[["hours_studied", "exam_score", "predicted_score", "error", "squared_error"]].round(2)
```

</details>

In [16]:
scores_1d["squared_error"] = scores_1d["error"] ** 2
scores_1d[["hours_studied", "exam_score", "predicted_score", "error", "squared_error"]].round(2)

,hours_studied,exam_score,predicted_score,error,squared_error
0,1.8,47.1,44.0,3.1,9.61
1,2.0,40.2,45.0,-4.8,23.04
2,2.8,55.7,49.0,6.7,44.89
3,4.0,56.7,55.0,1.7,2.89
4,4.5,58.6,57.5,1.1,1.21
5,4.5,55.7,57.5,-1.8,3.24
6,4.6,67.6,58.0,9.6,92.16
7,6.2,68.2,66.0,2.2,4.84
8,6.6,68.7,68.0,0.7,0.49
9,7.1,71.9,70.5,1.4,1.96


## Section 7 — Sum of Squared Errors (SSE)

$$SSE = \sum_{i=1}^{n} (y_i - \hat y_i)^2$$

```text
large SSE  = poor fit
smaller SSE = better fit
```

### ✏️ Try it yourself

Write a function `compute_sse(dataframe, slope, intercept)` that returns the SSE for a given line.

In [17]:
def compute_sse(dataframe, slope, intercept):
    # TODO: compute predictions, residuals, squared errors, and return their sum
    pass

<details>
<summary>Show solution</summary>

```python
def compute_sse(dataframe, slope, intercept):
    predictions = slope * dataframe["hours_studied"] + intercept
    residuals = dataframe["exam_score"] - predictions
    squared_errors = residuals ** 2
    return squared_errors.sum()

compute_sse(scores_1d, candidate_slope, candidate_intercept)
```

</details>

In [18]:
def compute_sse(dataframe, slope, intercept):
    predictions = slope * dataframe["hours_studied"] + intercept
    residuals = dataframe["exam_score"] - predictions
    squared_errors = residuals ** 2
    return squared_errors.sum()

current_sse = compute_sse(scores_1d, candidate_slope, candidate_intercept)
print(f"Current line: y = {candidate_slope}x + {candidate_intercept}")
print(f"Current SSE: {current_sse:.1f}")

Current line: y = 5.0x + 35.0
Current SSE: 519.1


## Section 8 — The Regression Game

Try different `slope` / `intercept` pairs below and watch the SSE. **You are training the regression model manually.**

> Can anyone get SSE below 500? Below 300? Below 200?

In [19]:
# Change these and rerun.
game_slope = 5.5
game_intercept = 35.0

game_sse = compute_sse(scores_1d, game_slope, game_intercept)
plot_line_over_data(scores_1d, game_slope, game_intercept,
                     title=f"y = {game_slope}x + {game_intercept}   |   SSE = {game_sse:.1f}").show()
print(f"Current SSE: {game_sse:.1f}")

Current SSE: 260.7


## Section 9 — Visualizing Residuals

One clean, reusable visualization: points, the candidate line, and every residual as a vertical segment, with rich hover info.

In [20]:
def plot_line_with_residuals(dataframe, slope, intercept, title=None):
    x = dataframe["hours_studied"]
    y = dataframe["exam_score"]
    predictions = slope * x + intercept
    residuals = y - predictions
    squared_residuals = residuals ** 2

    line_x = np.linspace(x.min() - 0.5, x.max() + 0.5, 50)

    figure = go.Figure()
    for xi, yi, pi, ri, sqi in zip(x, y, predictions, residuals, squared_residuals):
        figure.add_trace(
            go.Scatter(
                x=[xi, xi], y=[pi, yi], mode="lines",
                line=dict(color="#f59e0b", width=2, dash="dot"), showlegend=False,
                hovertemplate=(
                    f"x: {xi:.1f}<br>actual: {yi:.1f}<br>prediction: {pi:.1f}"
                    f"<br>residual: {ri:.1f}<br>squared residual: {sqi:.1f}<extra></extra>"
                ),
            )
        )
    figure.add_trace(
        go.Scatter(x=x, y=y, mode="markers", marker=dict(size=11, color="#2563eb"), name="actual")
    )
    figure.add_trace(
        go.Scatter(x=line_x, y=slope * line_x + intercept, mode="lines",
                    line=dict(color="#dc2626", width=3), name=f"y = {slope:.2f}x + {intercept:.2f}")
    )

    sse = float((residuals ** 2).sum())
    figure.update_layout(
        title=title or f"SSE = {sse:.1f}", xaxis_title="Hours studied", yaxis_title="Exam score",
        template="plotly_white", width=750, height=500,
    )
    return figure, sse

figure, sse = plot_line_with_residuals(scores_1d, candidate_slope, candidate_intercept)
figure.show()

Every point we've plotted so far is a **training observation** with a real, known score. But we fit a line to answer questions about students we haven't seen yet. Here's a helper for exactly that — it plots a new x on the line with dashed guide lines, distinct from the training data.

In [21]:
def plot_new_prediction(dataframe, slope, intercept, new_x, title=None):
    x = dataframe["hours_studied"]
    y = dataframe["exam_score"]
    predicted_y = slope * new_x + intercept

    line_x = np.linspace(min(x.min(), new_x) - 0.5, max(x.max(), new_x) + 0.5, 50)

    figure = go.Figure()
    figure.add_trace(
        go.Scatter(x=x, y=y, mode="markers", marker=dict(size=11, color="#2563eb"), name="training students")
    )
    figure.add_trace(
        go.Scatter(x=line_x, y=slope * line_x + intercept, mode="lines",
                    line=dict(color="#dc2626", width=3), name=f"y = {slope:.2f}x + {intercept:.2f}")
    )
    figure.add_trace(
        go.Scatter(x=[new_x, new_x], y=[0, predicted_y], mode="lines",
                    line=dict(color="#7c3aed", width=2, dash="dash"), showlegend=False)
    )
    figure.add_trace(
        go.Scatter(x=[0, new_x], y=[predicted_y, predicted_y], mode="lines",
                    line=dict(color="#7c3aed", width=2, dash="dash"), showlegend=False)
    )
    figure.add_trace(
        go.Scatter(x=[new_x], y=[predicted_y], mode="markers",
                    marker=dict(size=16, color="#7c3aed", symbol="star"),
                    name="new prediction (no actual value)",
                    hovertemplate=f"x: {new_x} (new / unseen)<br>predicted y: {predicted_y:.1f}<extra></extra>")
    )
    figure.update_layout(
        title=title or f"Predicting for an unseen x = {new_x}",
        xaxis_title="Hours studied", yaxis_title="Exam score", template="plotly_white", width=750, height=500,
    )
    return figure, predicted_y

demo_figure, demo_prediction = plot_new_prediction(scores_1d, candidate_slope, candidate_intercept, new_x=6.5)
demo_figure.show()

## Section 10 — What Are We Actually Searching For?

$$\hat y = mx + b \qquad L(m, b) = \sum_{i=1}^{n} (y_i - (mx_i + b))^2$$

The dataset is fixed. The only things we can change are `m` and `b`. Regression asks: which values of m and b produce the smallest error?

```text
(m, b) -> predictions -> residuals -> SSE
```

> ### 💭 Think about it
>
> - What exactly are we changing when we "train" this model?

## Section 11 — Visualize the Loss Landscape

SSE depends on `m` and `b`, so we can plot SSE as a surface over every possible (m, b) pair.

In [22]:
slope_range = np.linspace(-2, 12, 60)
intercept_range = np.linspace(0, 70, 60)
slope_grid, intercept_grid = np.meshgrid(slope_range, intercept_range)

x_values = scores_1d["hours_studied"].to_numpy()
y_values = scores_1d["exam_score"].to_numpy()
predictions_grid = slope_grid[..., None] * x_values + intercept_grid[..., None]
sse_grid = ((y_values - predictions_grid) ** 2).sum(axis=-1)

best_index = np.unravel_index(np.argmin(sse_grid), sse_grid.shape)
best_slope_grid = slope_grid[best_index]
best_intercept_grid = intercept_grid[best_index]
best_sse_grid = sse_grid[best_index]

surface_figure = go.Figure(data=[go.Surface(x=slope_grid, y=intercept_grid, z=sse_grid, colorscale="Viridis", opacity=0.9)])
surface_figure.add_trace(
    go.Scatter3d(x=[best_slope_grid], y=[best_intercept_grid], z=[best_sse_grid],
                 mode="markers", marker=dict(size=6, color="red"), name="minimum")
)
surface_figure.update_layout(
    title="SSE Loss Landscape",
    scene=dict(xaxis_title="slope (m)", yaxis_title="intercept (b)", zaxis_title="SSE"),
    width=800, height=600,
)
surface_figure.show()
print(f"Approximate minimum near: slope={best_slope_grid:.2f}, intercept={best_intercept_grid:.2f}, SSE={best_sse_grid:.1f}")

Approximate minimum near: slope=5.83, intercept=34.41, SSE=241.3


> ### 💭 Think about it
>
> - What would "training" mean on this surface?

## Section 12 — Connection to Calculus

At the bottom of the valley, the surface is momentarily flat in every direction:

$$\frac{\partial L}{\partial m} = 0 \qquad \frac{\partial L}{\partial b} = 0$$

```text
derivatives -> minimum -> model parameters
```

## Section 13 — Calculate the Best Line Manually

$$m = \frac{\sum (x_i - \bar x)(y_i - \bar y)}{\sum (x_i - \bar x)^2} \qquad b = \bar y - m \bar x$$

You don't need to memorize this — just understand what it finds: the exact (m, b) that minimizes SSE.

### ✏️ Try it yourself

Implement the formula above in clear, separate steps. Use variables `x_mean`, `y_mean`, `numerator`, `denominator`, then compute `manual_slope` and `manual_intercept`.

In [23]:
# TODO: x_mean, y_mean
# TODO: numerator = sum of (x_i - x_mean)(y_i - y_mean)
# TODO: denominator = sum of (x_i - x_mean)^2
# TODO: manual_slope, manual_intercept

<details>
<summary>Show solution</summary>

```python
x_mean = scores_1d["hours_studied"].mean()
y_mean = scores_1d["exam_score"].mean()

deviations_x = scores_1d["hours_studied"] - x_mean
deviations_y = scores_1d["exam_score"] - y_mean

numerator = (deviations_x * deviations_y).sum()
denominator = (deviations_x ** 2).sum()

manual_slope = numerator / denominator
manual_intercept = y_mean - manual_slope * x_mean

print(f"slope (m):     {manual_slope:.4f}")
print(f"intercept (b): {manual_intercept:.4f}")
```

</details>

In [24]:
x_mean = scores_1d["hours_studied"].mean()
y_mean = scores_1d["exam_score"].mean()
deviations_x = scores_1d["hours_studied"] - x_mean
deviations_y = scores_1d["exam_score"] - y_mean
numerator = (deviations_x * deviations_y).sum()
denominator = (deviations_x ** 2).sum()
manual_slope = numerator / denominator
manual_intercept = y_mean - manual_slope * x_mean

manual_sse = compute_sse(scores_1d, manual_slope, manual_intercept)
print(f"slope (m):     {manual_slope:.4f}")
print(f"intercept (b): {manual_intercept:.4f}")
print(f"SSE:           {manual_sse:.2f}")

figure, _ = plot_line_with_residuals(scores_1d, manual_slope, manual_intercept,
                                      title=f"Manual least-squares line | SSE = {manual_sse:.1f}")
figure.show()

slope (m):     5.8260
intercept (b): 34.0551
SSE:           239.03


## Section 14 — Finally Use Scikit-Learn

Everything sklearn is about to do, we already did by hand above.

### ✏️ Try it yourself

Fit `sklearn.linear_model.LinearRegression` on `scores_1d`. Remember: `X` must be 2D (shape `(n_samples, 1)`), `y` can be 1D. Print `model.coef_` and `model.intercept_`.

In [25]:
from sklearn.linear_model import LinearRegression

# TODO: build X (2D) and y (1D)
# TODO: create and fit the model
# TODO: print model.coef_ and model.intercept_

<details>
<summary>Show solution</summary>

```python
from sklearn.linear_model import LinearRegression

X = scores_1d[["hours_studied"]]
y = scores_1d["exam_score"]

model = LinearRegression()
model.fit(X, y)

print("model.coef_:     ", model.coef_)
print("model.intercept_:", model.intercept_)
```

</details>

In [26]:
from sklearn.linear_model import LinearRegression

X = scores_1d[["hours_studied"]]
y = scores_1d["exam_score"]

model = LinearRegression()
model.fit(X, y)

sklearn_slope = model.coef_[0]
sklearn_intercept = model.intercept_
sklearn_sse = compute_sse(scores_1d, sklearn_slope, sklearn_intercept)

comparison = pd.DataFrame({
    "Method": ["Human attempt", "Manual least squares", "sklearn"],
    "Slope": [game_slope, manual_slope, sklearn_slope],
    "Intercept": [game_intercept, manual_intercept, sklearn_intercept],
    "SSE": [game_sse, manual_sse, sklearn_sse],
})
comparison.round(4)

,Method,Slope,Intercept,SSE
0,Human attempt,5.50,35.00,260.68
1,Manual least squares,5.83,34.06,239.03
2,sklearn,5.83,34.06,239.03


The **manual least-squares** and **sklearn** rows should match to numerical precision. That's the "aha" moment: `model.fit()` solves the exact same problem you just solved by hand.

## Section 15 — Prediction

Back to our opening question: a new student studies **6.5 hours**. What score do we predict?

### ✏️ Try it yourself

Compute the prediction for `new_hours = 6.5` two ways: (1) manually using `sklearn_slope` and `sklearn_intercept`, (2) using `model.predict(...)`. They should match.

In [27]:
new_hours = 6.5

# TODO: manual_prediction using the line equation
# TODO: sklearn_prediction using model.predict

<details>
<summary>Show solution</summary>

```python
new_hours = 6.5

manual_prediction = sklearn_slope * new_hours + sklearn_intercept
sklearn_prediction = model.predict([[new_hours]])[0]

print(f"Manual prediction:  {manual_prediction:.2f}")
print(f"sklearn prediction: {sklearn_prediction:.2f}")
```

</details>

In [28]:
new_hours = 6.5
manual_prediction = sklearn_slope * new_hours + sklearn_intercept
sklearn_prediction = model.predict([[new_hours]])[0]
print(f"Manual prediction:  {manual_prediction:.2f}")
print(f"sklearn prediction: {sklearn_prediction:.2f}")

prediction_figure, _ = plot_new_prediction(scores_1d, sklearn_slope, sklearn_intercept, new_x=new_hours,
                                            title=f"Predicting for a new student who studied {new_hours} hours")
prediction_figure.show()

Manual prediction:  71.92
sklearn prediction: 71.92


/home/akbar/akbarDev/hbai/academy-tutorials/linear-regression/.venv/lib/python3.13/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(


```text
Training:   finding parameters     (model.fit)
Prediction: using those parameters (model.predict)
```

## Section 16 — Extra Practice

Same dataset, fresh attempt — a good way to check everything above actually stuck before moving to two features.

1. Pick your own slope and intercept, plot them.
2. Compute predictions, residuals, squared residuals, SSE.
3. Fit with sklearn and compare your SSE to sklearn's.
4. Predict the score for a student who studies 4 hours.

In [29]:
# 1. Plot your own line.
practice_slope = ...   # replace with a number
practice_intercept = ...   # replace with a number

<details>
<summary>Show solution</summary>

```python
practice_slope = 5.5
practice_intercept = 33.0
plot_line_over_data(scores_1d, practice_slope, practice_intercept).show()
```

</details>

In [30]:
# 2. Predictions, residuals, squared residuals, SSE.

<details>
<summary>Show solution</summary>

```python
practice_predictions = practice_slope * scores_1d["hours_studied"] + practice_intercept
practice_residuals = scores_1d["exam_score"] - practice_predictions
practice_squared_errors = practice_residuals ** 2
practice_sse = practice_squared_errors.sum()
print("Your SSE:", practice_sse)
```

</details>

In [31]:
# 3. Fit with sklearn and compare.

<details>
<summary>Show solution</summary>

```python
practice_model = LinearRegression()
practice_model.fit(scores_1d[["hours_studied"]], scores_1d["exam_score"])
practice_sklearn_sse = compute_sse(scores_1d, practice_model.coef_[0], practice_model.intercept_)

print("Your SSE:   ", practice_sse)
print("sklearn SSE:", practice_sklearn_sse)
```

</details>

In [32]:
# 4. Predict for 4 hours studied.

<details>
<summary>Show solution</summary>

```python
print(practice_model.predict([[4]]))
```

</details>

## What If One Point Is Way Off?

Back in Section 6 we mentioned that squaring errors means least squares can be sensitive to outliers. Let's actually see it.

Imagine one more student is added to our dataset: they studied for just **1 hour** but somehow scored **95**. Everyone else still looks the same.

In [33]:
outlier_row = pd.DataFrame([{"hours_studied": 1.0, "exam_score": 95.0}])
scores_1d_with_outlier = pd.concat([scores_1d, outlier_row], ignore_index=True)

outlier_model = LinearRegression()
outlier_model.fit(scores_1d_with_outlier[["hours_studied"]], scores_1d_with_outlier["exam_score"])
outlier_slope = outlier_model.coef_[0]
outlier_intercept = outlier_model.intercept_

line_x = np.linspace(scores_1d["hours_studied"].min() - 0.5, scores_1d["hours_studied"].max() + 0.5, 50)

figure = go.Figure()
figure.add_trace(
    go.Scatter(
        x=scores_1d["hours_studied"], y=scores_1d["exam_score"], mode="markers",
        marker=dict(size=11, color="#2563eb"), name="original students",
    )
)
figure.add_trace(
    go.Scatter(
        x=outlier_row["hours_studied"], y=outlier_row["exam_score"], mode="markers",
        marker=dict(size=16, color="#dc2626", symbol="diamond"), name="outlier",
    )
)
figure.add_trace(
    go.Scatter(
        x=line_x, y=sklearn_slope * line_x + sklearn_intercept, mode="lines",
        line=dict(color="#16a34a", width=3, dash="dash"), name="best fit WITHOUT outlier",
    )
)
figure.add_trace(
    go.Scatter(
        x=line_x, y=outlier_slope * line_x + outlier_intercept, mode="lines",
        line=dict(color="#dc2626", width=3), name="best fit WITH outlier",
    )
)
figure.update_layout(
    title="One Outlier, Two Very Different Lines",
    xaxis_title="Hours studied", yaxis_title="Exam score",
    template="plotly_white", width=750, height=500,
)
figure.show()

comparison_with_outlier = pd.DataFrame({
    "Fit": ["Without outlier", "With outlier"],
    "Slope": [sklearn_slope, outlier_slope],
    "Intercept": [sklearn_intercept, outlier_intercept],
    "SSE (on original 16 points)": [
        compute_sse(scores_1d, sklearn_slope, sklearn_intercept),
        compute_sse(scores_1d, outlier_slope, outlier_intercept),
    ],
})
comparison_with_outlier.round(3)

,Fit,Slope,Intercept,SSE (on original 16 points)
0,Without outlier,5.83,34.05,239.03
1,With outlier,3.34,50.78,821.19


A single point — one out of seventeen — pulled the slope down and the intercept up. The line minimizes squared error across *every* point, so one point with a huge error (before fitting) gets a huge say in where the line ends up.

> This isn't something we'll fix today (that's what robust regression techniques are for), but it's important to know least squares has this soft spot.

## Section 17 — The Big Question

> Real models normally don't only receive one piece of information. What happens if the prediction depends on **two** things?

$$\hat y = b + w_1 x_1 + w_2 x_2$$

```text
Previously:  one input  -> one coefficient -> a LINE
Now:         two inputs -> two coefficients -> a PLANE
```

In [34]:
scores_2d = pd.read_csv("../data/study_scores_2d.csv")
scores_2d.head(10)

,hours_studied,practice_problems,exam_score
0,6.0,20,87.7
1,8.2,0,63.1
2,7.2,2,77.4
3,2.8,4,53.2
4,3.4,20,68.1
5,8.0,14,87.4
6,1.0,18,65.0
7,7.6,4,69.1
8,7.4,15,89.9
9,4.7,7,61.3


## Section 18 — 3D Regression

Points only, first. Rotate the plot and look for where a flat surface would pass through the cloud.

In [35]:
scatter_3d = go.Figure(
    data=[go.Scatter3d(
        x=scores_2d["hours_studied"], y=scores_2d["practice_problems"], z=scores_2d["exam_score"],
        mode="markers", marker=dict(size=5, color="#2563eb"),
        hovertemplate="Hours: %{x}<br>Problems: %{y}<br>Score: %{z}<extra></extra>",
    )]
)
scatter_3d.update_layout(
    title="Exam Score vs Hours Studied and Practice Problems",
    scene=dict(xaxis_title="Hours studied", yaxis_title="Practice problems", zaxis_title="Exam score"),
    width=800, height=650,
)
scatter_3d.show()

> ### 💭 Think about it
>
> - Where would you put a flat surface that approximately passes through these points?

In [36]:
def add_regression_plane(figure, dataframe, intercept, w1, w2, opacity=0.5):
    x_range = np.linspace(dataframe["hours_studied"].min(), dataframe["hours_studied"].max(), 15)
    y_range = np.linspace(dataframe["practice_problems"].min(), dataframe["practice_problems"].max(), 15)
    x_grid, y_grid = np.meshgrid(x_range, y_range)
    z_grid = intercept + w1 * x_grid + w2 * y_grid
    figure.add_trace(go.Surface(x=x_grid, y=y_grid, z=z_grid, opacity=opacity, colorscale="Reds", showscale=False))
    return figure

quick_2d_model = LinearRegression()
quick_2d_model.fit(scores_2d[["hours_studied", "practice_problems"]], scores_2d["exam_score"])

plane_figure = go.Figure(
    data=[go.Scatter3d(
        x=scores_2d["hours_studied"], y=scores_2d["practice_problems"], z=scores_2d["exam_score"],
        mode="markers", marker=dict(size=5, color="#2563eb"),
    )]
)
add_regression_plane(plane_figure, scores_2d, quick_2d_model.intercept_, *quick_2d_model.coef_)
plane_figure.update_layout(
    title="Points with a Fitted Regression Plane",
    scene=dict(xaxis_title="Hours studied", yaxis_title="Practice problems", zaxis_title="Exam score"),
    width=800, height=650,
)
plane_figure.show()

## Section 19 — Manual Plane Fitting

$$\hat y = b + w_1 x_1 + w_2 x_2$$

Just like the line, but now three numbers to adjust: intercept `b`, hours coefficient `w1`, practice coefficient `w2`.

```text
1 feature  -> adjust a LINE
2 features -> adjust a PLANE
```

### ✏️ Try it yourself

Write `compute_sse_2d(dataframe, intercept, w1, w2)` that returns the SSE for a candidate plane.

In [ ]:
def compute_sse_2d(dataframe, intercept, w1, w2)linspace:
    # TODO: compute predictions, residuals, squared errors, and return their sum
    pass

<details>
<summary>Show solution</summary>

```python
def compute_sse_2d(dataframe, intercept, w1, w2):
    predictions = intercept + w1 * dataframe["hours_studied"] + w2 * dataframe["practice_problems"]
    residuals = dataframe["exam_score"] - predictions
    return (residuals ** 2).sum()
```

</details>

In [38]:
def compute_sse_2d(dataframe, intercept, w1, w2):
    predictions = intercept + w1 * dataframe["hours_studied"] + w2 * dataframe["practice_problems"]
    residuals = dataframe["exam_score"] - predictions
    return (residuals ** 2).sum()

# Try changing these three numbers.
my_intercept = 30.0
my_w1 = 4.0
my_w2 = 1.0

print("Your SSE:", compute_sse_2d(scores_2d, my_intercept, my_w1, my_w2))

Your SSE: 2926.560000000001


> ### 📝 Note
>
> For the full live-slider version of this exercise, switch to the Streamlit app: `streamlit run streamlit_app/app.py`, Tab 4 — Fit a Plane.

## Section 20 — Many Features: How Far Does This Idea Go?

```text
1 feature:   ŷ = b + w1*x1                     -> LINE
2 features:  ŷ = b + w1*x1 + w2*x2              -> PLANE
3 features:  ŷ = b + w1*x1 + w2*x2 + w3*x3      -> beyond what we can draw
```

With 40 features:

$$\hat y = b + \sum_{j=1}^{40} w_j x_j$$

### The design matrix

```text
rows    = observations (one row per student)
columns = features     (one column per input variable)
```

With 100 students and 40 features: `X.shape = (100, 40)`. The model learns 40 coefficients plus one intercept — but **every observation still produces exactly one prediction**.

> ### 💭 Think about it
>
> - What do you think replaces the line in three dimensions?
> - What happens if we have 40 inputs?

In [39]:
rng = np.random.default_rng(seed=1)
n_students = 1000
n_features = 40

X_wide = rng.normal(size=(n_students, n_features))
true_weights = rng.normal(scale=2.0, size=n_features)
y_wide = 50 + X_wide @ true_weights + rng.normal(scale=5, size=n_students)

wide_model = LinearRegression()
wide_model.fit(X_wide, y_wide)

print("X.shape:            ", X_wide.shape)
print("model.coef_.shape:  ", wide_model.coef_.shape)

X.shape:             (1000, 40)
model.coef_.shape:   (40,)


```text
x1  * w1
x2  * w2
...
x40 * w40
      |
     SUM
      +
  intercept
      |
  prediction
```

> Linear regression is called **linear** because these weighted feature contributions are added together linearly. The geometry becomes impossible to picture past 2-3 features, but the mathematical idea has not changed.

## Section 21 — Back to Two Features, Properly Named

Let's finish with our real 2-feature dataset and present the coefficients the way you'd actually want to read them.

### ✏️ Try it yourself

Fit a `LinearRegression` on `scores_2d[["hours_studied", "practice_problems"]]` predicting `exam_score`. Build a small dataframe with columns `feature` and `coefficient` (include the intercept, labeled `"Intercept"`, as the first row).

In [40]:
features = ["hours_studied", "practice_problems"]

# TODO: build X_final, y_final
# TODO: fit final_model
# TODO: build coefficient_table with columns "feature" and "coefficient"

<details>
<summary>Show solution</summary>

```python
features = ["hours_studied", "practice_problems"]
X_final = scores_2d[features]
y_final = scores_2d["exam_score"]

final_model = LinearRegression()
final_model.fit(X_final, y_final)

coefficient_table = pd.DataFrame({
    "feature": ["Intercept"] + features,
    "coefficient": [final_model.intercept_] + list(final_model.coef_),
})
coefficient_table.round(2)
```

</details>

In [41]:
features = ["hours_studied", "practice_problems"]
X_final = scores_2d[features]
y_final = scores_2d["exam_score"]

final_model = LinearRegression()
final_model.fit(X_final, y_final)

coefficient_table = pd.DataFrame({
    "feature": ["Intercept"] + features,
    "coefficient": [final_model.intercept_] + list(final_model.coef_),
})
coefficient_table.round(2)

,feature,coefficient
0,Intercept,30.72
1,hours_studied,4.68
2,practice_problems,1.34


### One more prediction, with two features — shown on the plane

In [42]:
new_hours_studied = 6
new_practice_problems = 10

manual_2d_prediction = (
    final_model.intercept_
    + final_model.coef_[0] * new_hours_studied
    + final_model.coef_[1] * new_practice_problems
)
sklearn_2d_prediction = final_model.predict([[new_hours_studied, new_practice_problems]])[0]

print(f"Manual prediction:  {manual_2d_prediction:.2f}")
print(f"sklearn prediction: {sklearn_2d_prediction:.2f}")

new_prediction_figure = go.Figure(
    data=[go.Scatter3d(
        x=scores_2d["hours_studied"], y=scores_2d["practice_problems"], z=scores_2d["exam_score"],
        mode="markers", marker=dict(size=5, color="#2563eb"), name="training students",
    )]
)
add_regression_plane(new_prediction_figure, scores_2d, final_model.intercept_, *final_model.coef_)
new_prediction_figure.add_trace(
    go.Scatter3d(
        x=[new_hours_studied], y=[new_practice_problems], z=[sklearn_2d_prediction],
        mode="markers", marker=dict(size=7, color="#7c3aed", symbol="diamond"),
        name="new prediction (no actual value)",
    )
)
new_prediction_figure.update_layout(
    title="Predicting for a new student, shown on the plane",
    scene=dict(xaxis_title="Hours studied", yaxis_title="Practice problems", zaxis_title="Exam score"),
    width=800, height=650,
)
new_prediction_figure.show()

Manual prediction:  72.23
sklearn prediction: 72.23


/home/akbar/akbarDev/hbai/academy-tutorials/linear-regression/.venv/lib/python3.13/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(


A 30-feature version works **exactly the same way** — the library simply receives more columns.

## What We Covered Today

```text
Prediction -> Residual -> Square -> Sum -> SSE -> Minimize -> Best coefficients
```

| Features | Equation | Visual |
|---|---|---|
| 1 | ŷ = b + w1x1 | LINE |
| 2 | ŷ = b + w1x1 + w2x2 | PLANE |
| many | ŷ = b + Σ wj xj | HYPERPLANE |

> Linear regression is not fundamentally about drawing a line. It is about learning coefficients that combine input features to produce predictions while minimizing prediction error. A line is simply the version we are lucky enough to visualize.

## Final Check

1. What is a residual?
2. Why don't we simply add raw residuals?
3. What does least squares try to minimize?
4. If one input produces a line, what do two inputs produce?
5. If a model has 40 input features, how can it still produce one prediction?

<details>
<summary>Show solution</summary>

```python
1. A residual is the difference between the actual observed value and the model's prediction: e_i = y_i - y_hat_i.
2. Raw residuals can be positive or negative and cancel out when summed, hiding how wrong the model actually is.
3. Least squares minimizes the sum of squared residuals (SSE) across all observations.
4. Two inputs produce a plane (in 3D); more generally, p inputs produce a hyperplane in (p+1)-dimensional space.
5. Each feature is multiplied by its learned coefficient, the contributions are added together with the
   intercept, and the result is one predicted value -- regardless of how many features go in.
```

</details>